# Data Modeling Task
The goal of this task is to examine the master census metrics JSON file for the CCSVI Dashboard and understand how to transition the existing data into a format usable by an LLM.

### Updates
- master JSON file: `public/data/metrics/census_metrics_by_block_group.json`.
- flattened the nested JSON into a long-form Pandas DataFrame.

# Inspecting the Master JSON

In [107]:
import json
import pandas as pd

In [108]:
# Loading the master .json file
master_file = "../public/data/metrics/census_metrics_by_block_group.json"
with open(master_file, "r") as f:
    data = json.load(f)

In [109]:
# Examining structure
sample_geoid = list(data.keys())[0]
sample = data[sample_geoid]

print("Sample GEOID:", sample_geoid)
print("\nTop-level keys:")
print(sample.keys())

print("\nGeographic Info:")
for key in ["type", "block_group", "census_tract", "county", "state", "population"]:
    print(f"{key}: {sample.get(key)}")

print("\nMetrics datasets:")
print(sample["metrics"].keys())

first_dataset = list(sample["metrics"].keys())[0]
print("\nSample metric names:")
print(list(sample["metrics"][first_dataset].keys())[:10])

Sample GEOID: 5003

Top-level keys:
dict_keys(['type', 'name', 'block_group', 'census_tract', 'county', 'state', 'population', 'metrics'])

Geographic Info:
type: hawaiian_homeland
block_group: None
census_tract: None
county: None
state: None
population: 257

Metrics datasets:
dict_keys(['2022_census_hawaiian_homelands'])

Sample metric names:
['Total Population Under 5', 'Total Population Under 18', 'Total Population Over 65', 'Total population SEX Male', 'Total population SEX Female', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race White', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Black or African American', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race American Indian and Alaska Native', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Asian', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Native Hawaiian and Other Pacific Islander']


In [110]:
# Pivot into long data frame
rows = []

for geoid, area_data in data.items():
    base_info = {
        "geoid": geoid,
        "type": area_data.get("type"),
        "name": area_data.get("name"),
        "block_group": area_data.get("block_group"),
        "census_tract": area_data.get("census_tract"),
        "county": area_data.get("county"),
        "state": area_data.get("state"),
        "population": area_data.get("population")
    }
    
    for dataset_name, metrics in area_data.get("metrics", {}).items():
        for metric_name, values in metrics.items():
            rows.append({
                **base_info,
                "dataset": dataset_name,
                "metric": metric_name,
                "absolute": values.get("absolute"),
                "proportion": values.get("proportion")
            })

df_long = pd.DataFrame(rows)
df_long.head()
print("Shape:", df_long.shape)
type_counts = df_long.groupby("type").size().reset_index(name="row_count")
type_counts

Shape: (44520, 12)


,type,row_count
0,block_group,43320
1,hawaiian_homeland,1200


In [115]:
# Compare metrics between block groups and Hawaiian homelands
metrics_block = set(df_long[df_long["type"] == "block_group"]["metric"].unique())
metrics_hh = set(df_long[df_long["type"] == "hawaiian_homeland"]["metric"].unique())

print("Total Block Group metrics:", len(metrics_block))
print("Total Hawaiian Homeland metrics:", len(metrics_hh))

# Metrics shared by both types
shared_metrics = metrics_block.intersection(metrics_hh)
print("\nShared metrics:", len(shared_metrics))
print(sorted(shared_metrics))

# Metrics unique to Block Groups
block_only = metrics_block - metrics_hh
print("\nMetrics only in Block Groups:", len(block_only))
print(sorted(block_only))

# Metrics unique to Hawaiian Homelands
hh_only = metrics_hh - metrics_block
print("\nMetrics only in Hawaiian Homelands:", len(hh_only))
print(sorted(hh_only))

Total Block Group metrics: 40
Total Hawaiian Homeland metrics: 16

Shared metrics: 0
[]

Metrics only in Block Groups: 40
['American Indian and Alaska Native alone', 'Asian alone', 'Asian and Pacific Island languages: Limited English speaking household', 'Black or African American alone', 'Estimate Aggregate number of vehicles available', 'Estimate Total', 'Female', 'Females Over 65', 'Females Under 18', 'Females Under 5', 'In households: Householder: Female: Living alone', 'In households: Householder: Male: Living alone', 'Institutionalized population', 'Institutionalized population: Correctional facilities for adults', 'Institutionalized population: Juvenile facilities', 'Institutionalized population: Nursing facilities/Skilled-nursing facilities', 'Institutionalized population: Other institutional facilities', 'Male', 'Males Over 65', 'Males Under 18', 'Males Under 5', 'Native Hawaiian and Other Pacific Islander alone', 'No Computer', 'No Health Insurance Coverage', 'No Internet acc

## Structural Observations

1. Two geographic types exist:
   - block_group (standard Census hierarchy — nested within census tracts and counties)
   - hawaiian_homeland (a separate geographic designation that does not follow the tract → block group nesting)

2. Metrics are nested by:
   GEOID → dataset → metric_name → {absolute, proportion}

3. Block groups and Hawaiian Homelands do not have the same metric coverage.
   - Block groups contain 40 metrics.
   - Hawaiian Homelands contain 16 metrics.
   - Some indicators (e.g., institutionalized population, internet access, vehicle access) only exist for block groups.

In [116]:
# Exploring a schema design
df_geo = (
    df_long[df_long["type"] == "block_group"]
    [["geoid", "block_group", "census_tract", "county", "state", "population"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_geo.head()
df_geo = pd.DataFrame(geo_rows)

df_geo.head()

,geoid,block_group,census_tract,county,state,population
0,150010201001,Block Group 1,Census Tract 201,Hawaii County,Hawaii,1826.0
1,150010201002,Block Group 2,Census Tract 201,Hawaii County,Hawaii,1044.0
2,150010201003,Block Group 3,Census Tract 201,Hawaii County,Hawaii,1350.0
3,150010201004,Block Group 4,Census Tract 201,Hawaii County,Hawaii,1213.0
4,150010202021,Block Group 1,Census Tract 202.02,Hawaii County,Hawaii,888.0


---
# Schema Design

Based on the structure of the master JSON, we can model the data into **3 relational tables**.
This mirrors how the data would be stored in a SQL database (e.g. PostgreSQL + PostGIS).

| Table | Description |
|---|---|
| `geographic_areas` | One row per GEOID — location and population info |
| `datasets` | Reference table for the 14 source metric datasets |
| `vulnerability_metrics` | Long-form fact table — every metric value per area |

> **Note on PostGIS:** A future `geometry` column on `geographic_areas` would store the actual
> polygon shape of each census block group (sourced from GeoJSON/GeoTIFF files).
> This would allow spatial queries like: *which high-poverty block groups intersect a flood hazard zone?*

## Table 1: `geographic_areas`
One row per GEOID. Captures the geographic hierarchy and population for each area.

In [ ]:
# Build geographic_areas table
geo_rows = []

for geoid, area in data.items():
    geo_rows.append({
        "geoid":        geoid,
        "type":         area.get("type"),
        "name":         area.get("name"),
        "block_group":  area.get("block_group"),
        "census_tract": area.get("census_tract"),
        "county":       area.get("county"),
        "state":        area.get("state"),
        "population":   area.get("population"),
        # geometry column placeholder — will be populated once GeoJSON/PostGIS is integrated
        # "geometry": None
    })

df_geo = pd.DataFrame(geo_rows)

print("Shape:", df_geo.shape)
print("\nCounts by type:")
print(df_geo["type"].value_counts())
print("\nCounts by county:")
print(df_geo["county"].value_counts())
df_geo.head()

## Table 2: `datasets`
A reference/lookup table for the 14 source datasets.
All block group datasets come from the ACS (American Community Survey).
The Hawaiian Homeland dataset is a separate 2022 Census designation.

In [ ]:
# Build datasets reference table
from collections import defaultdict

dataset_geo_type = defaultdict(set)
for geoid, area in data.items():
    for ds in area.get("metrics", {}).keys():
        dataset_geo_type[ds].add(area.get("type"))

dataset_rows = [
    {"dataset_id": ds, "geo_type": ", ".join(sorted(types))}
    for ds, types in sorted(dataset_geo_type.items())
]

df_datasets = pd.DataFrame(dataset_rows)
print("Total datasets:", len(df_datasets))
df_datasets

## Table 3: `vulnerability_metrics`
The core fact table. One row per (geoid, dataset, metric) combination.
Each metric has an `absolute` count and a `proportion` (share of population).

This is the table the LLM will query most heavily.

In [ ]:
# Build vulnerability_metrics table (long-form fact table)
metric_rows = []

for geoid, area in data.items():
    for dataset_id, metrics in area.get("metrics", {}).items():
        for metric_name, values in metrics.items():
            metric_rows.append({
                "geoid":      geoid,
                "dataset_id": dataset_id,
                "metric":     metric_name,
                "absolute":   values.get("absolute"),
                "proportion": values.get("proportion"),
            })

df_metrics = pd.DataFrame(metric_rows)

print("Shape:", df_metrics.shape)
print(f"\nNull absolute values:   {df_metrics['absolute'].isna().sum()}")
print(f"Null proportion values: {df_metrics['proportion'].isna().sum()}")
df_metrics.head(10)

## Schema Validation
Checking referential integrity — every metric row should have a matching GEOID in `geographic_areas`,
and every dataset_id should exist in the `datasets` reference table.

In [ ]:
# Validate: all GEOIDs in metrics exist in geographic_areas
geoids_in_geo     = set(df_geo["geoid"])
geoids_in_metrics = set(df_metrics["geoid"])
orphaned_geoids   = geoids_in_metrics - geoids_in_geo

print("=== GEOID Integrity ===")
print(f"GEOIDs in geographic_areas:      {len(geoids_in_geo)}")
print(f"GEOIDs in vulnerability_metrics: {len(geoids_in_metrics)}")
print(f"Orphaned GEOIDs:                 {len(orphaned_geoids)}")

# Validate: all dataset_ids in metrics exist in datasets reference table
ds_in_ref     = set(df_datasets["dataset_id"])
ds_in_metrics = set(df_metrics["dataset_id"])
missing_ds    = ds_in_metrics - ds_in_ref

print("\n=== Dataset ID Integrity ===")
print(f"Dataset IDs in reference table: {len(ds_in_ref)}")
print(f"Dataset IDs in metrics:         {len(ds_in_metrics)}")
print(f"Missing dataset IDs:            {len(missing_ds)}")

## SQL Schema (DDL)
This is what the schema would look like written as SQL — the language used to define
and query a relational database like PostgreSQL.

The commented-out `geometry` column is where **PostGIS** would come in:
storing the actual polygon boundary of each block group so we can run
spatial queries against climate hazard layers (e.g. flood zones, wildfire risk).

In [ ]:
sql_schema = """
-- Table 1: Geographic areas (census block groups + Hawaiian homelands)
CREATE TABLE geographic_areas (
    geoid           TEXT PRIMARY KEY,
    type            TEXT,              -- 'block_group' or 'hawaiian_homeland'
    name            TEXT,
    block_group     TEXT,
    census_tract    TEXT,
    county          TEXT,
    state           TEXT,
    population      INTEGER
    -- geometry    GEOMETRY(MULTIPOLYGON, 4326)  -- PostGIS: add once GeoJSON is integrated
);

-- Table 2: Dataset reference table
CREATE TABLE datasets (
    dataset_id      TEXT PRIMARY KEY,
    geo_type        TEXT               -- 'block_group' or 'hawaiian_homeland'
);

-- Table 3: Vulnerability metrics (fact table)
CREATE TABLE vulnerability_metrics (
    geoid           TEXT REFERENCES geographic_areas(geoid),
    dataset_id      TEXT REFERENCES datasets(dataset_id),
    metric          TEXT,
    absolute        FLOAT,
    proportion      FLOAT,
    PRIMARY KEY (geoid, dataset_id, metric)
);

-- Example query: top 10 block groups by uninsured rate
-- SELECT g.geoid, g.county, g.population, m.proportion
-- FROM vulnerability_metrics m
-- JOIN geographic_areas g ON m.geoid = g.geoid
-- WHERE m.metric = 'No Health Insurance Coverage'
-- ORDER BY m.proportion DESC
-- LIMIT 10;
"""

print(sql_schema)

## Schema Observations

1. **1,158 total geographic areas** — 1,083 block groups across 5 Hawaii counties (Hawaii, Honolulu, Kalawao, Kauai, Maui), and 75 Hawaiian homelands.

2. **14 source datasets** — 13 apply only to block groups (ACS data), 1 applies only to Hawaiian homelands (`2022_census_hawaiian_homelands`). There is zero metric overlap between the two geographic types, so the LLM will need to be aware of which type it is querying.

3. **44,520 total metric rows** — with some null values present (~2,600 null absolutes, ~3,900 null proportions). These will need handling before feeding to an LLM (e.g. fill with 0, drop, or flag as missing).

4. **Key vulnerability signals for LLM queries:**
   - `health_insurance` → No Health Insurance Coverage
   - `income_share_of_fpl` → poverty thresholds at 100%, 150%, 200% FPL
   - `limited_english_speaking` → language access barriers
   - `internet_subscription` / `households_w_computer` → digital access gap
   - `tenure` → Renter occupied (housing instability proxy)
   - `population_group_quarters` → institutionalized populations

5. **PostGIS integration (future):** Adding a `geometry` column to `geographic_areas` would enable spatial joins with climate hazard layers (flood zones, wildfire risk, sea level rise, etc.) — the core goal of the CCSVI dashboard pipeline.